# PRM800K — Real Dataset Hypothesis Test

**Goal:** Train a Process Reward Model (PRM) using OpenAI's real step-level labels from PRM800K,  
then evaluate whether Gemma-2B's reasoning errors are detectable at the step level.

---

## 🔒 Cell-by-Cell Safety Design
Every cell is **idempotent** — you can re-run any single cell without reloading expensive resources.  
Guards like `if 'model' not in dir()` and disk-based caching prevent redundant work.

| Cell | What it does | Re-runnable? |
|------|-------------|-------------|
| 1 | Auth | ✅ Safe |
| 2 | Install deps | ✅ Safe |
| 3 | Clone PRM800K repo | ✅ Skips if already cloned |
| 4 | Load model & tokenizer | ✅ Skips if already in memory |
| 5 | Load GSM8K | ✅ Skips if already loaded |
| 6 | Load PRM800K labels | ✅ Skips if already loaded |
| 7 | Generate GSM8K solutions | ✅ Loads from disk if cached |
| 8 | Embed steps | ✅ Loads from disk if cached |
| 9 | Train classifier | ✅ Always fast |
| 10 | GO/NO-GO Verdict | ✅ Always fast |
| 11 | Spot-check failures | ✅ Always fast |

## Cell 1 — Authentication
You need a Hugging Face token with access to `google/gemma-2b-it`.  
Store it as a Colab secret named `HF_TOKENN` via the 🔑 key icon in the left sidebar.

In [1]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKENN'))
print("✅ Hugging Face login successful.")

✅ Hugging Face login successful.


## Cell 2 — Install Dependencies
Safe to re-run. `git-lfs` is required to download the large PRM800K `.jsonl` files.

In [2]:
# Install git-lfs for large file downloads from the PRM800K repo
!apt-get install -y git-lfs -q
# !pip install -q transformers accelerate datasets scikit-learn tqdm
print("✅ Dependencies ready.")

Reading package lists...
Building dependency tree...
Reading state information...
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Dependencies ready.


## Cell 3 — Clone OpenAI PRM800K Repository
**Only clones once.** If the folder already exists, this cell is skipped entirely.  
The real dataset lives in `prm800k/data/` as `.jsonl` files.

In [3]:
import os

REPO_DIR  = "/content/prm800k"
DATA_DIR  = f"{REPO_DIR}/prm800k/data"

if os.path.exists(DATA_DIR):
    print("✅ PRM800K repo already cloned — skipping.")
    print(f"   Files: {os.listdir(DATA_DIR)}")
else:
    print("📥 Cloning PRM800K repo (this may take a few minutes for LFS files)...")
    !git lfs install --skip-repo -q
    !git clone https://github.com/openai/prm800k.git {REPO_DIR}
    # Pull the actual LFS data files
    !cd {REPO_DIR} && git lfs pull
    if os.path.exists(DATA_DIR):
        print(f"✅ Clone complete. Files: {os.listdir(DATA_DIR)}")
    else:
        print("⚠️  Data directory not found after clone — check repo structure.")
        !find {REPO_DIR} -name '*.jsonl' | head -20

📥 Cloning PRM800K repo (this may take a few minutes for LFS files)...
Error: unknown shorthand flag: 'q' in -q
git lfs install [options]

Perform the following actions to ensure that Git LFS is setup properly:

* Set up the clean and smudge filters under the name "lfs" in the global Git
  config.
* Install a pre-push hook to run git lfs pre-push for the current repository,
  if run from inside one. If "core.hooksPath" is configured in any Git
  configuration (and supported, i.e., the installed Git version is at least
  2.9.0), then the pre-push hook will be installed to that directory instead.
  
Options:

Without any options, git lfs install will only setup the "lfs" smudge and clean
filters if they are not already set.

* --force:
    Sets the "lfs" smudge and clean filters, overwriting existing values.
* --local:
    Sets the "lfs" smudge and clean filters in the local repository's git
    config, instead of the global git config (~/.gitconfig).
* --worktree:
    Sets the "lfs" smud

## Cell 4 — Load Gemma-2B Model & Tokenizer
**Guard:** Skips loading if `model` is already in memory.  
This is the most expensive step (~5 min on T4). Only runs on first execution.

In [4]:
import torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "google/gemma-2b-it"

print(f"Device: {DEVICE}")

# ── GUARD: skip if already loaded ──────────────────────────────────────────
if 'model' in dir() and model is not None:
    print("✅ Model already loaded — skipping.")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"📥 Loading {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    print("✅ Gemma-2B loaded and ready.")

Device: cuda
📥 Loading google/gemma-2b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Gemma-2B loaded and ready.


## Cell 5 — Load GSM8K Dataset
**Guard:** Skips if already loaded.  
We use the test split — 1,319 problems with verified numeric answers.

In [5]:
from datasets import load_dataset

# ── GUARD: skip if already loaded ──────────────────────────────────────────
if 'gsm8k' in dir() and gsm8k is not None:
    print(f"✅ GSM8K already loaded — {len(gsm8k)} problems.")
else:
    print("📥 Loading GSM8K test split...")
    gsm8k = load_dataset("gsm8k", "main", split="test")
    print(f"✅ Loaded {len(gsm8k)} GSM8K test problems.")

📥 Loading GSM8K test split...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ Loaded 1319 GSM8K test problems.


## Cell 6 — Load REAL PRM800K Step-Level Labels
**This is the core replacement for the proxy label approach.**

Instead of labeling every step by final answer correctness, we load **human-annotated step labels** directly from the PRM800K dataset.

**Rating values in the real data:**
- `+1` → Correct step
- ` 0` → Neutral / ambiguous
- `-1` → Incorrect step

**Guard:** Skips if already loaded.

In [7]:
import json
import os

# ── GUARD: skip if already loaded ──────────────────────────────────────────
if 'prm_texts' in dir() and len(prm_texts) > 0:
    print(f"✅ PRM800K labels already loaded — {len(prm_texts)} steps.")
else:
    DATA_DIR = "/content/prm800k/prm800k/data"

    PHASE_FILES = [
        "phase1_train.jsonl",
        "phase2_train.jsonl",
    ]

    prm_texts    = []
    prm_labels   = []
    prm_problems = []
    rating_counts = {1: 0, 0: 0, -1: 0}
    skipped_null  = 0   # counter so you can see how many null records exist

    for fname in PHASE_FILES:
        fpath = os.path.join(DATA_DIR, fname)
        if not os.path.exists(fpath):
            print(f"⚠️  {fname} not found — skipping.")
            continue

        count_before = len(prm_texts)
        with open(fpath, "r") as f:
            for line in f:
                item = json.loads(line.strip())
                problem_text = (item.get("question") or {}).get("problem", "")

                # ── FIX: use `or []` to handle explicit null values ──────
                for step in (item.get("label") or {}).get("steps") or []:
                    for completion in (step.get("completions") or []):
                        text   = completion.get("text", "").strip()
                        rating = completion.get("rating", None)

                        if not text or rating is None:
                            skipped_null += 1
                            continue

                        if rating in rating_counts:
                            rating_counts[rating] += 1

                        label = 1 if rating == 1 else 0

                        prm_texts.append(text)
                        prm_labels.append(label)
                        prm_problems.append(problem_text)

        print(f"   {fname}: loaded {len(prm_texts) - count_before:,} step completions")

    print(f"\n✅ PRM800K loaded: {len(prm_texts):,} total step-level labels")
    print(f"   Skipped null/empty entries: {skipped_null:,}")
    print(f"   Raw ratings  → +1 (correct): {rating_counts[1]:,} | "
          f"0 (neutral): {rating_counts[0]:,} | "
          f"-1 (wrong): {rating_counts[-1]:,}")
    print(f"   Binary split → Correct (1): {sum(prm_labels):,} | "
          f"Wrong (0): {len(prm_labels)-sum(prm_labels):,}")

✅ PRM800K labels already loaded — 7227 steps.


## Cell 7 — Generate Gemma-2B Solutions on GSM8K

**Guard:** Results are saved to `/content/gsm8k_results.json` after first run.  
Re-running this cell loads from disk instantly — no GPU time wasted.

**What happens here:**  
Gemma-2B generates step-by-step solutions. We record whether each solution was correct.  
This gives us real model outputs to spot-check against the PRM in Cell 11.

In [8]:
import re
import json
import os
from tqdm import tqdm

RESULTS_CACHE = "/content/gsm8k_results.json"
N_PROBLEMS    = 150   # increase for more thorough testing
MAX_NEW_TOKENS = 300

# ── Helpers ────────────────────────────────────────────────────────────────
def extract_answer(text):
    match = re.search(r'Answer:\s*([0-9,.\-]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    nums = re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else None

def extract_steps(text):
    steps = re.findall(
        r'(Step\s*\d+[:\.].*?)(?=Step\s*\d+[:\.]|Answer:|$)',
        text, re.DOTALL
    )
    return [s.strip() for s in steps if len(s.strip()) > 10]

def get_ground_truth(answer_text):
    nums = re.findall(r'####\s*([0-9,.\-]+)', answer_text)
    if nums:
        return nums[-1].replace(',', '').strip()
    nums = re.findall(r'-?\d+\.?\d*', answer_text)
    return nums[-1] if nums else None

COT_PROMPT = """Solve the following math problem step by step.
Label each step as 'Step 1:', 'Step 2:', etc.
End with 'Answer: <number>'.

Problem: {problem}

Solution:"""

# ── GUARD: load from disk if cached ────────────────────────────────────────
if os.path.exists(RESULTS_CACHE):
    with open(RESULTS_CACHE, "r") as f:
        results = json.load(f)
    n_correct = sum(r["is_correct"] for r in results)
    base_accuracy = n_correct / len(results)
    print(f"✅ Loaded {len(results)} results from disk cache.")
    print(f"   Gemma-2B accuracy: {base_accuracy:.1%}  "
          f"(Correct: {n_correct} | Wrong: {len(results)-n_correct})")
else:
    print(f"⚙️  Generating solutions for {N_PROBLEMS} GSM8K problems...")
    print("   This takes ~20–30 min on T4. Will be cached after completion.\n")

    subset  = gsm8k.select(range(min(N_PROBLEMS, len(gsm8k))))
    results = []

    for i, example in tqdm(enumerate(subset), total=len(subset), desc="Generating"):
        problem     = example["question"]
        correct_ans = get_ground_truth(example["answer"])
        prompt      = COT_PROMPT.format(problem=problem)

        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        pred_ans = extract_answer(generated)
        steps    = extract_steps(generated)
        correct  = (pred_ans == correct_ans) if pred_ans and correct_ans else False

        results.append({
            "problem":        problem,
            "generated":      generated,
            "steps":          steps,
            "pred_answer":    pred_ans,
            "correct_answer": correct_ans,
            "is_correct":     correct,
        })

        if (i + 1) % 25 == 0:
            running_acc = sum(r["is_correct"] for r in results) / len(results)
            print(f"  [{i+1}/{len(subset)}] running accuracy: {running_acc:.1%}")

    # Save to disk
    with open(RESULTS_CACHE, "w") as f:
        json.dump(results, f)

    base_accuracy = sum(r["is_correct"] for r in results) / len(results)
    n_correct     = sum(r["is_correct"] for r in results)
    print(f"\n✅ Done. Cached to {RESULTS_CACHE}")
    print(f"   Gemma-2B accuracy: {base_accuracy:.1%}  "
          f"(Correct: {n_correct} | Wrong: {len(results)-n_correct})")

⚙️  Generating solutions for 150 GSM8K problems...
   This takes ~20–30 min on T4. Will be cached after completion.



Generating:  17%|█▋        | 25/150 [01:51<07:46,  3.73s/it]

  [25/150] running accuracy: 4.0%


Generating:  33%|███▎      | 50/150 [03:42<06:15,  3.76s/it]

  [50/150] running accuracy: 10.0%


...

  [100/150] running accuracy: 17.0%


Generating:  83%|████████▎ | 125/150 [08:44<01:24,  3.40s/it]

  [125/150] running accuracy: 16.0%


Generating: 100%|██████████| 150/150 [10:32<00:00,  4.22s/it]

  [150/150] running accuracy: 16.0%

✅ Done. Cached to /content/gsm8k_results.json
   Gemma-2B accuracy: 16.0%  (Correct: 24 | Wrong: 126)


## Cell 8 — Embed Steps Using Gemma-2B Hidden States

**Source of embeddings:** We use the **real PRM800K step texts** (not GSM8K-generated steps)  
to train the classifier. This is the key difference from the proxy-label approach.

**Guard:** Embeddings are saved to `/content/prm_embeddings.npz`.  
Re-running loads from disk — no re-embedding needed.

**Why mean-pool last hidden state?**  
Gemma-2B already understands mathematical reasoning. Its internal representations  
encode whether a step is logically consistent — we're extracting that signal directly.

In [9]:
import numpy as np
import os
from tqdm import tqdm

EMB_CACHE     = "/content/prm_embeddings.npz"
MAX_EMBED     = 5000    # number of PRM800K steps to embed (balanced)
MAX_LENGTH    = 128     # token truncation per step
BATCH_SIZE    = 8

# ── GUARD: load from disk if cached ────────────────────────────────────────
if os.path.exists(EMB_CACHE):
    data = np.load(EMB_CACHE, allow_pickle=True)
    X = data["X"]
    y = data["y"]
    print(f"✅ Loaded embeddings from disk: {X.shape[0]:,} steps, dim={X.shape[1]}")
    print(f"   Correct (1): {y.sum():,} | Wrong (0): {(y==0).sum():,}")
else:
    # Balance classes from PRM800K
    pos_idx = [i for i, l in enumerate(prm_labels) if l == 1]
    neg_idx = [i for i, l in enumerate(prm_labels) if l == 0]
    n_each  = min(len(pos_idx), len(neg_idx), MAX_EMBED // 2)

    np.random.seed(42)
    selected_pos = np.random.choice(pos_idx, n_each, replace=False)
    selected_neg = np.random.choice(neg_idx, n_each, replace=False)
    selected_idx = np.concatenate([selected_pos, selected_neg])
    np.random.shuffle(selected_idx)

    embed_texts  = [prm_texts[i]  for i in selected_idx]
    embed_labels = [prm_labels[i] for i in selected_idx]

    print(f"⚙️  Embedding {len(embed_texts):,} balanced PRM800K steps "
          f"({n_each:,} correct + {n_each:,} wrong)...")

    embeddings = []
    for i in tqdm(range(0, len(embed_texts), BATCH_SIZE), desc="Embedding"):
        batch = embed_texts[i : i + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        hidden = outputs.hidden_states[-1]                     # (B, seq, dim)
        mask   = inputs["attention_mask"].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1)          # mean pool
        embeddings.extend(emb.cpu().float().numpy())

    X = np.array(embeddings)
    y = np.array(embed_labels)

    # Save to disk
    np.savez(EMB_CACHE, X=X, y=y)
    print(f"\n✅ Embeddings saved to {EMB_CACHE}")
    print(f"   Shape: {X.shape} | Correct: {y.sum():,} | Wrong: {(y==0).sum():,}")

⚙️  Embedding 5,000 balanced PRM800K steps (2,500 correct + 2,500 wrong)...


Embedding: 100%|██████████| 625/625 [01:29<00:00,  6.95it/s]



✅ Embeddings saved to /content/prm_embeddings.npz
   Shape: (5000, 2048) | Correct: 2,500 | Wrong: 2,500


## Cell 9 — Train PRM Classifier

Logistic regression on Gemma-2B's hidden states.  
Fast to train (seconds). Always re-trains from the loaded embeddings.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Train / test split (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training on {len(X_train):,} steps, evaluating on {len(X_test):,} steps...")

clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
clf.fit(X_train, y_train)

y_pred       = clf.predict(X_test)
prm_accuracy = accuracy_score(y_test, y_pred)

print(f"\n{'='*55}")
print(f"  PRM Step-Level Accuracy:  {prm_accuracy:.1%}")
print(f"{'='*55}")
print("\nDetailed Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Wrong step (0)", "Correct step (1)"]
))

Training on 3,750 steps, evaluating on 1,250 steps...

  PRM Step-Level Accuracy:  66.3%

Detailed Classification Report:
                  precision    recall  f1-score   support

  Wrong step (0)       0.66      0.68      0.67       625
Correct step (1)       0.67      0.65      0.66       625

        accuracy                           0.66      1250
       macro avg       0.66      0.66      0.66      1250
    weighted avg       0.66      0.66      0.66      1250



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Cell 10 — GO / NO-GO Verdict

The verdict uses **real PRM800K accuracy** — not proxy-label accuracy.  
This means the threshold is meaningful: we're asking whether Gemma-2B's  
hidden states encode step correctness *as judged by human annotators*.

In [11]:
print("=" * 60)
print("  SLD-VM PRM  —  REAL PRM800K  —  GO / NO-GO VERDICT")
print("=" * 60)
print(f"  Dataset used:              OpenAI PRM800K (real labels)")
print(f"  Step-level labels:         Human-annotated (+1 / 0 / -1)")
print(f"  Training steps:            {len(X_train):,}")
print(f"  Test steps:                {len(X_test):,}")
print(f"  Gemma-2B GSM8K accuracy:   {base_accuracy:.1%}")
print(f"  PRM step-level accuracy:   {prm_accuracy:.1%}")
print()

if prm_accuracy >= 0.80:
    verdict = "✅  GO"
    msg = (
        "PRM reliably detects Gemma-2B reasoning errors at the step level.\n"
        "  Gemma-2B's hidden states encode step validity as judged by\n"
        "  human annotators. VCE pipeline is viable — commit fully."
    )
elif prm_accuracy >= 0.70:
    verdict = "⚠️   CONDITIONAL GO"
    msg = (
        "PRM shows meaningful signal but is not fully reliable yet.\n"
        "  Next steps:\n"
        "    (1) Increase MAX_EMBED (more training data)\n"
        "    (2) Try a dedicated 125M verifier (DeBERTa/BERT)\n"
        "    (3) Use phase2 labels only (higher quality annotations)"
    )
else:
    verdict = "❌  NO-GO"
    msg = (
        "Gemma-2B hidden states do not reliably encode step correctness\n"
        "  as defined by human PRM800K annotators.\n"
        "  Consider: dedicated verifier model, rule-based checks, or\n"
        "  a larger base model before pursuing this pipeline."
    )

print(f"  VERDICT:  {verdict}")
print()
print(f"  {msg}")
print("=" * 60)

  SLD-VM PRM  —  REAL PRM800K  —  GO / NO-GO VERDICT
  Dataset used:              OpenAI PRM800K (real labels)
  Step-level labels:         Human-annotated (+1 / 0 / -1)
  Training steps:            3,750
  Test steps:                1,250
  Gemma-2B GSM8K accuracy:   16.0%
  PRM step-level accuracy:   66.3%

  VERDICT:  ❌  NO-GO

  Gemma-2B hidden states do not reliably encode step correctness
  as defined by human PRM800K annotators.
  Consider: dedicated verifier model, rule-based checks, or
  a larger base model before pursuing this pipeline.


## Cell 11 — Spot-Check: Does the PRM Flag Gemma-2B's Real Errors?

This cell takes the **Gemma-2B outputs from Cell 7** (wrong solutions)  
and runs the PRM classifier on each step to see if it flags the errors.  

This is the real-world validation: can the PRM trained on PRM800K labels  
detect mistakes in a **completely different model's (Gemma-2B's) outputs**?

In [12]:
import numpy as np

N_SPOT = 5  # number of wrong problems to inspect

def embed_single(text, max_length=128):
    """Embed a single step text using Gemma-2B last hidden state."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True,
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden = outputs.hidden_states[-1]
    mask   = inputs["attention_mask"].unsqueeze(-1).float()
    emb    = (hidden * mask).sum(1) / mask.sum(1)
    return emb.squeeze(0).cpu().float().numpy()

wrong_results = [r for r in results if not r["is_correct"]][:N_SPOT]

print(f"Spot-checking {len(wrong_results)} wrong solutions from Gemma-2B")
print(f"PRM was trained on PRM800K human labels — completely held-out domain\n")
print("=" * 60)

total_steps  = 0
flagged_steps = 0

for i, r in enumerate(wrong_results):
    print(f"\nProblem {i+1}: {r['problem'][:90]}...")
    print(f"  Correct answer : {r['correct_answer']}")
    print(f"  Model answer   : {r['pred_answer']}  ← WRONG")
    print(f"  Steps generated: {len(r['steps'])}")

    if not r["steps"]:
        print("  (no steps extracted)")
        continue

    for j, step in enumerate(r["steps"][:5]):  # show up to 5 steps
        total_steps += 1
        emb  = embed_single(step).reshape(1, -1)
        prob_wrong   = clf.predict_proba(emb)[0][0]   # probability of being WRONG
        prob_correct = clf.predict_proba(emb)[0][1]   # probability of being correct
        flagged      = prob_wrong > 0.55
        if flagged:
            flagged_steps += 1
        flag_str = "⚠️  FLAGGED" if flagged else "   ok     "
        print(f"    Step {j+1} [{flag_str}  wrong-prob={prob_wrong:.2f}]: {step[:75]}...")

print("\n" + "=" * 60)
if total_steps > 0:
    flag_rate = flagged_steps / total_steps
    print(f"  Steps analysed : {total_steps}")
    print(f"  Steps flagged  : {flagged_steps}  ({flag_rate:.1%} flag rate on wrong solutions)")
    print()
    if flag_rate >= 0.5:
        print("  ✅ PRM is catching errors in Gemma-2B's wrong solutions.")
        print("     The cross-domain transfer from PRM800K labels is working.")
    elif flag_rate >= 0.3:
        print("  ⚠️  PRM is catching some errors but missing others.")
        print("     Transfer from PRM800K to Gemma-2B is partial.")
    else:
        print("  ❌ PRM is not flagging errors in wrong solutions.")
        print("     The PRM800K signal does not transfer to Gemma-2B's error pattern.")

Spot-checking 5 wrong solutions from Gemma-2B
PRM was trained on PRM800K human labels — completely held-out domain


Problem 1: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes mu...
  Correct answer : 18
  Model answer   : 26  ← WRONG
  Steps generated: 5
    Step 1 [   ok       wrong-prob=0.00]: Step 1:** Janet lays 16 eggs per day.

**...
    Step 2 [   ok       wrong-prob=0.00]: Step 2:** She eats 3 eggs for breakfast every morning.

**...
    Step 3 [   ok       wrong-prob=0.00]: Step 3:** She bakes muffins for her friends every day with 4 eggs.

**...
    Step 4 [   ok       wrong-prob=0.02]: Step 4:** She sells the remainder at the farmers' market daily for $2 per f...
    Step 5 [   ok       wrong-prob=0.14]: Step 5:** To find the total amount she makes each day, we need to add the n...

Problem 2: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in tota...
  Correct answer : 3
  Model answer   : 4  ← WRON

## Cell 12 — Summary & Interpretation Guide

Understanding what the results mean for your hypothesis.

In [13]:
print("=" * 60)
print("  FULL EXPERIMENT SUMMARY")
print("=" * 60)
print()
print("HYPOTHESIS BEING TESTED:")
print("  Gemma-2B's internal representations (hidden states) are")
print("  sufficient to classify individual reasoning steps as")
print("  correct or incorrect, using PRM800K human-annotated labels.")
print()
print("WHAT CHANGED vs. YOUR ORIGINAL NOTEBOOK:")
print("  OLD  → Labels from final answer correctness (solution-level proxy)")
print("  NEW  → Labels from human annotators (step-level ground truth)")
print()
print("RESULTS:")
print(f"  Gemma-2B baseline accuracy (GSM8K) : {base_accuracy:.1%}")
print(f"  PRM step-level accuracy (PRM800K)  : {prm_accuracy:.1%}")
print(f"  Final verdict                      : {verdict}")
print()
print("INTERPRETING PRM ACCURACY:")
print("  > 80% → Hidden states encode step quality. VCE pipeline viable.")
print("  70-80% → Weak signal. Try more data or a dedicated verifier model.")
print("  < 70% → No meaningful signal. Rethink architecture.")
print()
print("CACHE FILES (safe to delete to reset):")
print("  /content/gsm8k_results.json   — Gemma-2B GSM8K solutions")
print("  /content/prm_embeddings.npz   — PRM800K step embeddings")
print("=" * 60)

  FULL EXPERIMENT SUMMARY

HYPOTHESIS BEING TESTED:
  Gemma-2B's internal representations (hidden states) are
  sufficient to classify individual reasoning steps as
  correct or incorrect, using PRM800K human-annotated labels.

WHAT CHANGED vs. YOUR ORIGINAL NOTEBOOK:
  OLD  → Labels from final answer correctness (solution-level proxy)
  NEW  → Labels from human annotators (step-level ground truth)

RESULTS:
  Gemma-2B baseline accuracy (GSM8K) : 16.0%
  PRM step-level accuracy (PRM800K)  : 66.3%
  Final verdict                      : ❌  NO-GO

INTERPRETING PRM ACCURACY:
  > 80% → Hidden states encode step quality. VCE pipeline viable.
  70-80% → Weak signal. Try more data or a dedicated verifier model.
  < 70% → No meaningful signal. Rethink architecture.

CACHE FILES (safe to delete to reset):
  /content/gsm8k_results.json   — Gemma-2B GSM8K solutions
  /content/prm_embeddings.npz   — PRM800K step embeddings
